In [ ]:
# ==========================================
# 1) تثبيت المكتبات اللازمة
# ==========================================
!pip install transformers datasets sentencepiece gdown scikit-learn -q --upgrade

# ==========================================
# 2) تحميل الملف من Google Drive (JSON)
# ==========================================
import os
import gdown
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

# رابط Google Drive المباشر
url = "https://drive.google.com/uc?id=12c_xh_awg6gNl_n4U9-voikOG44y_p4g"
json_path = os.path.join(DATA_DIR, "data.json")

# تحميل الملف
gdown.download(url, json_path, quiet=False)
print("✅ تم تحميل الملف بنجاح إلى:", json_path)

# ==========================================
# 3) قراءة البيانات وتقسيمها
# ==========================================
data = pd.read_json(json_path)

print("📊 معاينة أول صفوف:")
print(data.head())

# توحيد أسماء الأعمدة
data = data.rename(columns={
    "question": "input_text",
    "answer": "target_text"
})

# تقسيم البيانات إلى تدريب وتقييم
train_df, val_df = train_test_split(data, test_size=0.1, random_state=42)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

datasets = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

print("✅ عدد عينات التدريب:", len(train_dataset))
print("✅ عدد عينات التقييم:", len(val_dataset))

# ==========================================
# 4) تحميل النموذج والمُرمّز (Tokenizer)
# ==========================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

MODEL_NAME = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

# ==========================================
# 5) تجهيز البيانات للنموذج
# ==========================================
def preprocess_function(examples):
    inputs = [str(i) for i in examples["input_text"]]
    targets = [str(t) for t in examples["target_text"]]
    model_inputs = tokenizer(inputs, max_length=256, truncation=True)
    labels = tokenizer(targets, max_length=256, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = datasets.map(preprocess_function, batched=True)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# ==========================================
# 6) إعداد TrainingArguments
# ==========================================
from transformers import TrainingArguments, Trainer

OUTPUT_DIR = "/content/fatwa_t5_checkpoints"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=False,
    num_train_epochs=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=False,
    fp16=False,
    dataloader_num_workers=2,
    report_to=[]
)

# ==========================================
# 7) إنشاء الـ Trainer
# ==========================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("✅ Trainer جاهز — سيحفظ الملفات داخل:", OUTPUT_DIR)

# ==========================================
# 8) تدريب النموذج
# ==========================================
resume = False  # لو تريد الاستئناف من checkpoint سابق اجعلها True

if resume:
    import os
    last = None
    ckpts = sorted([d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint")],
                   key=lambda x: int(x.split("-")[-1]) if "-" in x else 0)
    if ckpts:
        last = os.path.join(OUTPUT_DIR, ckpts[-1])
        print("⚠ سيتم استئناف التدريب من:", last)
    trainer.train(resume_from_checkpoint=last)
else:
    trainer.train()

# ==========================================
# 9) حفظ النموذج النهائي والتوكنيزر داخل /content
# ==========================================
FINAL_DIR = "/content/fatwa_t5_model_final"
os.makedirs(FINAL_DIR, exist_ok=True)
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("✅ النموذج النهائي حفظ في:", FINAL_DIR)

# ضغط النموذج في ملف ZIP للتحميل
import shutil
zip_path = "/content/fatwa_t5_model_final.zip"
shutil.make_archive("/content/fatwa_t5_model_final", 'zip', FINAL_DIR)
print(f"📦 تم ضغط النموذج في: {zip_path}")

# رابط تحميل مباشر من Colab
from google.colab import files
files.download(zip_path)

# ==========================================
# 10) عرض الـ checkpoints الموجودة
# ==========================================
print("\n📁 Checkpoints الموجودة في", OUTPUT_DIR)
for name in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", name)

# ==========================================
# 11) حساب وعرض مصفوفة الارتباك (Confusion Matrix)
# ==========================================
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

print("\n📈 حساب مصفوفة الارتباك...")

predictions = trainer.predict(tokenized_datasets["validation"])
decoded_preds = tokenizer.batch_decode(predictions.predictions, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(predictions.label_ids, skip_special_tokens=True)

unique_labels = list(set(decoded_labels + decoded_preds))
label_to_id = {label: idx for idx, label in enumerate(unique_labels)}

y_true = [label_to_id[label] for label in decoded_labels]
y_pred = [label_to_id[pred] for pred in decoded_preds]

cm = confusion_matrix(y_true, y_pred, labels=range(len(unique_labels)))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=unique_labels)
fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, xticks_rotation=90, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - Validation Set")
plt.show()

print("✅ تم حساب وعرض مصفوفة الارتباك بنجاح.")

# ==========================================
# 12) حساب الدقة الإجمالية (Accuracy)
# ==========================================
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_true, y_pred)
print(f"🎯 دقة النموذج على مجموعة التقييم: {accuracy * 100:.2f}%")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 14.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.


Downloading...
From (original): https://drive.google.com/uc?id=12c_xh_awg6gNl_n4U9-voikOG44y_p4g
From (redirected): https://drive.google.com/uc?id=12c_xh_awg6gNl_n4U9-voikOG44y_p4g&confirm=t&uuid=b02fbd8f-028b-44de-a74a-2a7523efc135
To: /content/data/data.json
100%|██████████| 219M/219M [00:02<00:00, 86.6MB/s]


✅ تم تحميل الملف بنجاح إلى: /content/data/data.json
📊 معاينة أول صفوف:
                                            question  \
0     لماذا يضرب الله الأمثال لنفسه في القرآن بخلقه؟   
1               هل يرى الأطفال الرضع الملائكة والجن؟   
2                 هل وجود حكة في اليد دليل حصول رزق؟   
3                    هل المدبر من أسماء الله الحسنى؟   
4  ما الحكمة من إعطاء الله تعالى من لا يؤمن به ول...   

                                              answer  
0  أولا:يقول العلامة محمد الخضر حسين، رحمه الله: ...  
1  أولا:رؤية الملائكة، في صور غير صورهم الحقيقية ...  
2  لا علاقة لحكة اليد بقدوم رزق أو حصول شيء جميل،...  
3  وصف الله عز وجل بأنه المدبر لا إشكال فيه، ولا ...  
4  من أصول الإيمان أن يعلم العبد أن الله هو الرزا...  
✅ عدد عينات التدريب: 83699
✅ عدد عينات التقييم: 9300


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

...

Map:   0%|          | 0/9300 [00:00<?, ? examples/s]

/tmp/ipython-input-1322775671.py:107: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


✅ Trainer جاهز — سيحفظ الملفات داخل: /content/fatwa_t5_checkpoints


Step,Training Loss,Validation Loss
500,0.175900,0.158117
1000,0.161400,0.149433
1500,0.155700,0.146472
2000,0.156400,0.141539
2500,0.149000,0.138702
3000,0.145000,0.136828
3500,0.145200,0.136238
4000,0.151600,0.136272
4500,0.140000,0.133195
5000,0.136200,0.133727


Step,Training Loss,Validation Loss
500,0.175900,0.158117
1000,0.161400,0.149433
1500,0.155700,0.146472
2000,0.156400,0.141539
2500,0.149000,0.138702
3000,0.145000,0.136828
3500,0.145200,0.136238
4000,0.151600,0.136272
4500,0.140000,0.133195
5000,0.136200,0.133727


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
